# Estudo de Caso 3.1 — Balanço de Massa em Reator CSTR: Regime Transiente e Permanente

| Campo | Detalhe |
|:------|:--------|
| **Capítulo Associado** | Capítulo 3 — Modelagem Matemática do Escoamento de Fluidos |
| **Nível** | Graduação |
| **Tempo estimado** | 45 minutos |
| **Autor** | Jader Lugon Junior |
| **Método numérico** | EDO 1ª ordem — Solução analítica + Euler explícito |
| **Bibliografia** | Lugon Jr. et al. (2026), Cap. 3; Schnoor (1996), Cap. 2; Fogler (2020), Cap. 4 |

---

## 📋 Resumo

Aplicamos o Teorema do Transporte de Reynolds a um reator químico contínuo de tanque agitado (CSTR) para modelar a evolução temporal da concentração de um poluente em degradação de primeira ordem. A equação diferencial ordinária resultante é resolvida analiticamente e numericamente (método de Euler explícito), permitindo comparar as soluções e analisar a convergência do esquema. A análise de sensibilidade investiga o efeito da constante de reação e do tempo de residência hidráulica na concentração de regime permanente.

---

## 🎯 Objetivos de Aprendizagem

Ao final deste estudo de caso, o leitor será capaz de:

1. **Aplicar** o Teorema do Transporte de Reynolds a um volume de controle bem misturado
2. **Formular** a EDO de balanço de massa para um reator CSTR com reação de primeira ordem
3. **Resolver** analiticamente a EDO linear de primeira ordem com termo fonte
4. **Implementar** o método de Euler explícito e comparar com a solução exata
5. **Analisar** o critério de estabilidade do esquema de Euler para EDOs lineares
6. **Interpretar** fisicamente o tempo de residência hidráulica e o regime permanente
7. **Avaliar** a sensibilidade da concentração de equilíbrio aos parâmetros $k$ e $\tau$

---
## ⚙️ Seção 0: Configuração Inicial

In [ ]:
# ============================================================
# CONFIGURAÇÃO INICIAL — Importação de bibliotecas
# ============================================================

import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from scipy.integrate import solve_ivp
from IPython.display import display, Markdown
import time

# Configuração de estilo dos gráficos (padrão do guia)
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 11

print('✅ Ambiente configurado com sucesso')
print(f'📦 NumPy {np.__version__} | SciPy {__import__("scipy").__version__}')

---
## 🧪 Seção 1: Parâmetros do Reator

### Contexto Industrial

Um reator CSTR (*Continuous Stirred Tank Reactor*) é amplamente utilizado na engenharia ambiental para tratamento de efluentes. O reator é perfeitamente misturado, de modo que a concentração é uniforme em todo o volume. Um poluente entra continuamente e é degradado por reação de primeira ordem.

> 💡 **Observação:** A analogia da **conta bancária** é útil: o saldo (concentração no reator) muda pela diferença entre depósitos (entrada), saques (saída) e juros negativos (reação de degradação).

### Tabela de Parâmetros

| Parâmetro | Símbolo | Valor | Unidade |
|:----------|:-------:|------:|:-------:|
| Volume do reator | $V$ | 1000 | L |
| Vazão volumétrica | $Q$ | 10 | L/min |
| Concentração de entrada | $C_{in}$ | 50 | mg/L |
| Concentração inicial | $C_0$ | 0 | mg/L |
| Constante de reação (1ª ordem) | $k$ | 0,1 | min⁻¹ |
| Tempo de simulação | $t_{max}$ | 300 | min |

In [ ]:
# ============================================================
# PARÂMETROS DO REATOR CSTR
# ============================================================

# --- Geometria e operação ---
V = 1000.0          # Volume do reator [L]
Q = 10.0            # Vazão volumétrica [L/min]

# --- Concentrações ---
C_in = 50.0         # Concentração de entrada [mg/L]
C_0 = 0.0           # Concentração inicial no reator [mg/L]

# --- Cinética de reação ---
k = 0.1             # Constante de reação de 1ª ordem [min⁻¹]

# --- Parâmetros derivados ---
tau = V / Q         # Tempo de residência hidráulica [min]

# --- Simulação ---
t_max = 300.0       # Tempo total de simulação [min]

# --- Concentração de regime permanente (solução analítica) ---
C_ss = C_in / (1.0 + k * tau)   # [mg/L]

print('📏 Parâmetros do Reator CSTR:')
print(f'  • V = {V:.0f} L | Q = {Q:.0f} L/min')
print(f'  • C_in = {C_in} mg/L | C₀ = {C_0} mg/L')
print(f'  • k = {k} min⁻¹')
print(f'  • τ = V/Q = {tau:.0f} min')
print(f'  • C_ss = C_in/(1 + k·τ) = {C_ss:.2f} mg/L')

---
## 🔬 Seção 2: Verificação — Solução Analítica de Referência

Antes da implementação numérica, derivamos a solução analítica exata para servir como referência de validação.

A EDO linear de primeira ordem:

$$\frac{dC}{dt} + a \cdot C = b$$

com $a = \frac{1}{\tau} + k$ e $b = \frac{C_{in}}{\tau}$, tem solução exata:

$$C(t) = C_{ss} + (C_0 - C_{ss})\,e^{-a\,t}$$

onde $C_{ss} = b/a = \dfrac{C_{in}}{1 + k\tau}$.

> 📖 **Referência:** Lugon Jr. et al. (2026), Cap. 3, Seção 3.2.

In [ ]:
# ============================================================
# SOLUÇÃO ANALÍTICA EXATA
# ============================================================

t_inicio = time.time()

# Coeficiente de decaimento
a = 1.0 / tau + k   # [min⁻¹]

# Função da solução analítica
def C_analitica(t):
    """
    Solução analítica da EDO do reator CSTR.

    Parameters
    ----------
    t : float ou ndarray
        Tempo [min].

    Returns
    -------
    float ou ndarray
        Concentração C(t) [mg/L].

    Notes
    -----
    C(t) = C_ss + (C_0 - C_ss) * exp(-a*t)
    """
    return C_ss + (C_0 - C_ss) * np.exp(-a * t)

# Verificação em pontos-chave
t_test = np.array([0, tau, 2*tau, 3*tau, 5*tau])
C_test = C_analitica(t_test)

t_cpu_ana = time.time() - t_inicio

print('🔬 VERIFICAÇÃO — SOLUÇÃO ANALÍTICA')
print('=' * 50)
print(f'  • a = 1/τ + k = {a:.4f} min⁻¹')
print(f'  • C_ss = {C_ss:.2f} mg/L')
print(f'  • C(0) = {C_analitica(0):.2f} mg/L (deve ser C₀={C_0})')
print(f'\n📊 Evolução temporal:')
print(f'{"t [min]":<12}{"t/τ":<8}{"C(t) [mg/L]":<15}{"C/C_ss":<10}')
print('-' * 45)
for ti, Ci in zip(t_test, C_test):
    print(f'{ti:<12.0f}{ti/tau:<8.0f}{Ci:<15.2f}{Ci/C_ss:<10.2f}')
print(f'\n✅ Verificação: C(0)={C_analitica(0):.1f} mg/L, C(∞)={C_ss:.2f} mg/L')
print(f'⏱️ Tempo de CPU: {t_cpu_ana:.6f} s')

---
## 📐 Seção 3: Formulação Matemática

### 3.1 Teorema do Transporte de Reynolds

Para um volume de controle fixo (reator):

$$\frac{d}{dt}\int_{VC} C\,dV = -\oint_{SC} C(\mathbf{u} \cdot \mathbf{n})\,dA + \dot{S}$$

onde:
- $C$ = concentração do poluente [mg/L]
- $\mathbf{u}$ = velocidade do fluido [m/s]
- $\dot{S}$ = termo fonte/sumidouro [mg/min]

### 3.2 Hipótese de Mistura Perfeita

Para um CSTR ideal (concentração uniforme):

$$V\frac{dC}{dt} = Q \cdot C_{in} - Q \cdot C - k \cdot V \cdot C$$

### 3.3 Forma Normalizada

Dividindo por $V$ e definindo $\tau = V/Q$:

$$\frac{dC}{dt} = \frac{C_{in}}{\tau} - \left(\frac{1}{\tau} + k\right) C$$

### 3.4 Interpretação dos Termos

| Termo | Significado Físico |
|:------|:-------------------|
| $C_{in}/\tau$ | Entrada de poluente por convecção |
| $-C/\tau$ | Saída de poluente por convecção |
| $-kC$ | Degradação por reação de 1ª ordem |

> 📖 **Referência:** Lugon Jr. et al. (2026), Cap. 3, Seções 3.1–3.3; Schnoor (1996), Cap. 2.

---
## ⚡ Seção 4: Critério Numérico e Estabilidade

### Método de Euler Explícito

$$C^{n+1} = C^n + \Delta t \cdot f(t^n, C^n)$$

onde $f(t, C) = C_{in}/\tau - a \cdot C$.

### Condição de Estabilidade

Para a EDO linear $dC/dt = -aC + b$, o método de Euler explícito é estável se:

$$|1 - a\,\Delta t| \leq 1 \quad \Rightarrow \quad \Delta t \leq \frac{2}{a}$$

### Critério de Adequação

$$\Delta t \leq \frac{2}{1/\tau + k} = \frac{2\tau}{1 + k\tau}$$

> 💡 **Observação:** Para $\tau = 100$ min e $k = 0{,}1$ min⁻¹, temos $a = 0{,}02$ min⁻¹ e $\Delta t_{max} = 100$ min. Usaremos $\Delta t = 1$ min para alta precisão.

In [ ]:
# ============================================================
# VERIFICAÇÃO DO CRITÉRIO NUMÉRICO
# ============================================================

# Passo de tempo
dt = 1.0   # [min]

# Critério de estabilidade
dt_max = 2.0 / a   # [min]

# Número de passos
N_steps = int(t_max / dt)

# Número de Courant (análogo para EDO)
r = a * dt   # [-]

print('⚡ CRITÉRIO NUMÉRICO E ESTABILIDADE')
print('=' * 50)
print(f'  • a = {a:.4f} min⁻¹')
print(f'  • Δt escolhido: {dt} min')
print(f'  • Δt máximo (estabilidade): {dt_max:.0f} min')
print(f'  • Razão a·Δt = {r:.4f} (deve ser ≤ 2)')
print(f'  • Número de passos: {N_steps}')

if dt <= dt_max:
    print(f'\n  ✅ Esquema ESTÁVEL (Δt = {dt} min ≤ {dt_max:.0f} min)')
else:
    print(f'\n  ❌ Esquema INSTÁVEL! Reduzir Δt.')

print(f'\n  📏 Tempo de residência: τ = {tau:.0f} min')
print(f'  📏 Em t = 5τ = {5*tau:.0f} min: C = {C_ss + (C_0-C_ss)*np.exp(-5)*100/C_ss:.1f}% de C_ss')

---
## 🌪️ Seção 5: Condição de Entrada (Forçamento)

Neste problema, o forçamento é a **concentração de entrada constante**:

$$C_{in} = 50 \text{ mg/L (constante)}$$

com condição inicial:

$$C(0) = C_0 = 0 \text{ mg/L}$$

> 💡 **Observação:** O problema modela a **partida** do reator: inicialmente limpo ($C_0 = 0$), o poluente começa a entrar e acumular até atingir o regime permanente. Em aplicações reais, a concentração de entrada pode variar com o tempo (hidrograma de poluentes).

---
## 🧮 Seção 6: Implementação Numérica

In [ ]:
# ============================================================
# SOLUÇÃO NUMÉRICA — MÉTODO DE EULER EXPLÍCITO
# ============================================================

t_inicio = time.time()

# Vetores de tempo e concentração
t_num = np.linspace(0, t_max, N_steps + 1)   # [min]
C_num = np.zeros(N_steps + 1)                 # [mg/L]
C_num[0] = C_0

# Loop temporal — Euler explícito
for n in range(N_steps):
    # Termo fonte da EDO
    f = C_in / tau - a * C_num[n]
    # Atualização
    C_num[n + 1] = C_num[n] + dt * f

t_cpu_euler = time.time() - t_inicio

print('🧮 SOLUÇÃO NUMÉRICA — EULER EXPLÍCITO')
print('=' * 50)
print(f'  • Δt = {dt} min')
print(f'  • N_steps = {N_steps}')
print(f'  • C(0) = {C_num[0]:.2f} mg/L')
print(f'  • C(t_max) = {C_num[-1]:.2f} mg/L')
print(f'  • C_ss analítico = {C_ss:.2f} mg/L')
print(f'  • Erro final: {abs(C_num[-1] - C_ss):.4f} mg/L')
print(f'\n⏱️ Tempo de CPU: {t_cpu_euler:.6f} s')

In [ ]:
# ============================================================
# COMPARAÇÃO: EULER vs. SOLUÇÃO ANALÍTICA vs. RK45 (SciPy)
# ============================================================

# Solução com Runge-Kutta 4-5 (referência de alta ordem)
def dCdt(t, C):
    """Lado direito da EDO do reator CSTR."""
    return C_in / tau - a * C

t_inicio = time.time()
sol_rk = solve_ivp(dCdt, [0, t_max], [C_0], method='RK45',
                   t_eval=t_num, rtol=1e-10, atol=1e-12)
t_cpu_rk = time.time() - t_inicio

# Solução analítica nos mesmos pontos
C_ana = C_analitica(t_num)

# Erros
erro_euler = np.abs(C_num - C_ana)
erro_rk = np.abs(sol_rk.y[0] - C_ana)

print('📊 COMPARAÇÃO DE MÉTODOS')
print('=' * 50)
print(f'  • Euler (Δt={dt} min): erro máx = {np.max(erro_euler):.6f} mg/L')
print(f'  • RK45 (SciPy): erro máx = {np.max(erro_rk):.2e} mg/L')
print(f'  • Tempo CPU Euler: {t_cpu_euler:.6f} s')
print(f'  • Tempo CPU RK45: {t_cpu_rk:.6f} s')
print(f'\n✅ Ambos os métodos convergem para C_ss = {C_ss:.2f} mg/L')

---
## 📊 Seção 7: Visualização

In [ ]:
# ============================================================
# GRÁFICO — EVOLUÇÃO TEMPORAL DA CONCENTRAÇÃO
# ============================================================

fig, ax = plt.subplots(figsize=(12, 6))

# Solução analítica (curva contínua)
ax.plot(t_num, C_ana, 'k-', linewidth=2.5, label='Solução analítica')

# Solução numérica (Euler)
ax.plot(t_num[::5], C_num[::5], 'ro', markersize=4, alpha=0.7,
        label=f'Euler explícito (Δt={dt:.0f} min)')

# Regime permanente
ax.axhline(y=C_ss, color='green', linestyle='--', linewidth=1.5,
           label=f'Regime permanente: C_ss = {C_ss:.2f} mg/L')

# Marcar múltiplos de τ
for mult in [1, 2, 3]:
    ax.axvline(x=mult*tau, color='gray', linestyle=':', alpha=0.5)
    ax.text(mult*tau, C_ss*0.1, f'{mult}τ', ha='center', fontsize=9, color='gray')

ax.set_xlabel('Tempo t [min]', fontsize=12)
ax.set_ylabel('Concentração C [mg/L]', fontsize=12)
ax.set_title('Evolução Temporal da Concentração no Reator CSTR',
             fontsize=14, fontweight='bold')
ax.legend(fontsize=10, loc='center right')
ax.grid(True, alpha=0.3)
ax.set_xlim(0, t_max)
ax.set_ylim(0, C_in * 1.05)

plt.tight_layout()
plt.show()

> 💡 **Observação:**
> - A concentração cresce exponencialmente de zero até o regime permanente.
> - Após 1τ (100 min), a concentração atinge ~63% do valor final.
> - Após 3τ (300 min), atinge ~95% do valor final.
> - Após 5τ (500 min), atinge ~99,3% do valor final.
> - O método de Euler com Δt = 1 min é praticamente indistinguível da solução analítica.

In [ ]:
# ============================================================
# GRÁFICO — ERRO NUMÉRICO
# ============================================================

fig, ax = plt.subplots(figsize=(12, 5))

ax.semilogy(t_num, erro_euler + 1e-16, 'b-', linewidth=1.5,
            label=f'Euler (Δt={dt:.0f} min)')
ax.semilogy(t_num, erro_rk + 1e-16, 'r-', linewidth=1.5,
            label='RK45 (SciPy)')

ax.set_xlabel('Tempo t [min]', fontsize=12)
ax.set_ylabel('Erro absoluto [mg/L]', fontsize=12)
ax.set_title('Erro Absoluto vs. Solução Analítica',
             fontsize=14, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
ax.set_xlim(0, t_max)

plt.tight_layout()
plt.show()

> 💡 **Observação:**
> - O erro do Euler é máximo no início (onde o gradiente é maior) e decai com o tempo.
> - O erro do RK45 está no nível de precisão de máquina (~10⁻¹² mg/L).
> - Para Δt = 1 min, o erro máximo do Euler é < 0,01 mg/L (< 0,03% de C_ss).

---
## 📈 Seção 8: Análise Quantitativa

In [ ]:
# ============================================================
# TABELA COMPARATIVA DOS MÉTODOS
# ============================================================

dados_tabela = {
    'Característica': [
        'Método',
        'Passo de tempo',
        'Número de passos',
        'Tempo de CPU',
        'Erro máximo',
        'Erro relativo máximo',
        'Ordem de precisão',
        'Indicação de uso'
    ],
    'Euler Explícito': [
        'Euler',
        f'{dt:.0f} min',
        f'{N_steps}',
        f'{t_cpu_euler:.6f} s',
        f'{np.max(erro_euler):.6f} mg/L',
        f'{np.max(erro_euler)/C_ss*100:.4f}%',
        '1ª ordem',
        'Problemas simples, Δt pequeno'
    ],
    'Runge-Kutta 4-5': [
        'RK45',
        'Adaptativo',
        f'{sol_rk.t.size}',
        f'{t_cpu_rk:.6f} s',
        f'{np.max(erro_rk):.2e} mg/L',
        f'{np.max(erro_rk)/C_ss*100:.2e}%',
        '4ª-5ª ordem',
        'Problemas gerais, alta precisão'
    ]
}

df_comp = pd.DataFrame(dados_tabela)
display(Markdown(df_comp.to_markdown(index=False)))

In [ ]:
# ============================================================
# MÉTRICAS DE REGIME PERMANENTE
# ============================================================

# Tempo para atingir 95% de C_ss
t_95 = -np.log(0.05) / a   # [min]

# Tempo para atingir 99% de C_ss
t_99 = -np.log(0.01) / a   # [min]

# Eficiência de remoção em regime permanente
eficiencia = (1.0 - C_ss / C_in) * 100.0   # [%]

# Número de Damköhler (razão reação/convecção)
Da = k * tau   # [-]

print('📈 MÉTRICAS DE REGIME PERMANENTE')
print('=' * 50)
print(f'  • C_ss = {C_ss:.2f} mg/L ({C_ss/C_in*100:.1f}% de C_in)')
print(f'  • Eficiência de remoção: {eficiencia:.1f}%')
print(f'  • Número de Damköhler: Da = k·τ = {Da:.1f}')
print(f'  • Tempo para 63% de C_ss: t = τ = {tau:.0f} min')
print(f'  • Tempo para 95% de C_ss: t = {t_95:.0f} min ({t_95/tau:.1f}τ)')
print(f'  • Tempo para 99% de C_ss: t = {t_99:.0f} min ({t_99/tau:.1f}τ)')

---
## 🔍 Seção 9: Análise de Sensibilidade

Analisamos a sensibilidade da concentração de regime permanente a dois fatores:
1. **Constante de reação $k$:** efeito da velocidade de degradação
2. **Tempo de residência $\tau$:** efeito do dimensionamento do reator

In [ ]:
# ============================================================
# SENSIBILIDADE — Constante de reação k e tempo de residência τ
# ============================================================

# Variação de k (mantendo τ fixo)
k_range = np.linspace(0.001, 0.5, 100)
C_ss_k = C_in / (1.0 + k_range * tau)

# Variação de τ (mantendo k fixo)
tau_range = np.linspace(10, 500, 100)
C_ss_tau = C_in / (1.0 + k * tau_range)

# Gráfico
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Painel 1: C_ss vs k
ax1.plot(k_range, C_ss_k, 'b-', linewidth=2.5)
ax1.axvline(x=k, color='r', linestyle='--', linewidth=1.5,
            label=f'Caso base: k = {k} min⁻¹')
ax1.axhline(y=C_ss, color='g', linestyle=':', linewidth=1.5,
            label=f'C_ss = {C_ss:.1f} mg/L')
ax1.set_xlabel('Constante de reação k [min⁻¹]', fontsize=12)
ax1.set_ylabel('C_ss [mg/L]', fontsize=12)
ax1.set_title('(a) Efeito da Constante de Reação', fontsize=13, fontweight='bold')
ax1.legend(fontsize=10)
ax1.grid(True, alpha=0.3)

# Painel 2: C_ss vs τ
ax2.plot(tau_range, C_ss_tau, 'g-', linewidth=2.5)
ax2.axvline(x=tau, color='r', linestyle='--', linewidth=1.5,
            label=f'Caso base: τ = {tau:.0f} min')
ax2.axhline(y=C_ss, color='b', linestyle=':', linewidth=1.5,
            label=f'C_ss = {C_ss:.1f} mg/L')
ax2.set_xlabel('Tempo de residência τ [min]', fontsize=12)
ax2.set_ylabel('C_ss [mg/L]', fontsize=12)
ax2.set_title('(b) Efeito do Tempo de Residência', fontsize=13, fontweight='bold')
ax2.legend(fontsize=10)
ax2.grid(True, alpha=0.3)

plt.suptitle('Análise de Sensibilidade — Regime Permanente',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print(f'\n📊 Interpretação:')
print(f'  • k → ∞: C_ss → 0 (remoção total)')
print(f'  • τ → ∞: C_ss → 0 (tempo infinito de residência)')
print(f'  • k = 0: C_ss → C_in = {C_in} mg/L (sem reação)')
print(f'  • Da = k·τ = {Da:.1f}: para Da >> 1, remoção é eficiente')

In [ ]:
# ============================================================
# SENSIBILIDADE — Efeito do passo de tempo Δt
# ============================================================

dt_list = [0.5, 1.0, 5.0, 10.0, 50.0]
erros_max = []

for dt_test in dt_list:
    N_test = int(t_max / dt_test)
    t_test_arr = np.linspace(0, t_max, N_test + 1)
    C_test_arr = np.zeros(N_test + 1)
    C_test_arr[0] = C_0

    for n in range(N_test):
        f_test = C_in / tau - a * C_test_arr[n]
        C_test_arr[n + 1] = C_test_arr[n] + dt_test * f_test

    C_ana_test = C_analitica(t_test_arr)
    erros_max.append(np.max(np.abs(C_test_arr - C_ana_test)))

# Gráfico
fig, ax = plt.subplots(figsize=(10, 5))
ax.loglog(dt_list, erros_max, 'o-', color='red', linewidth=2, markersize=8)
# Linha de referência O(Δt)
dt_ref = np.array(dt_list)
erro_ref = erros_max[0] * dt_ref / dt_list[0]
ax.loglog(dt_ref, erro_ref, 'k--', linewidth=1.5, alpha=0.7,
          label='Referência: $\\mathcal{O}(\\Delta t)$')
ax.axvline(x=dt_max, color='gray', linestyle=':', linewidth=1.5,
           label=f'Limite de estabilidade: Δt = {dt_max:.0f} min')
ax.set_xlabel('Passo de tempo Δt [min]', fontsize=12)
ax.set_ylabel('Erro máximo [mg/L]', fontsize=12)
ax.set_title('Convergência do Método de Euler', fontsize=14, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3, which='both')

plt.tight_layout()
plt.show()

print(f'\n📊 Convergência de 1ª ordem confirmada:')
print(f'  • Ao dobrar Δt, o erro dobra (inclinação ≈ 1 em log-log)')
print(f'  • Para Δt > {dt_max:.0f} min, o esquema diverge')

> 💡 **Observação:**
> - O método de Euler é de **primeira ordem**: erro proporcional a $\Delta t$.
> - Para reduzir o erro por um fator 10, é necessário reduzir $\Delta t$ por um fator 10.
> - O RK45 (4ª-5ª ordem) é muito mais eficiente para alta precisão.
> - O limite de estabilidade $\Delta t_{max} = 2/a = 100$ min é generoso para este problema.

---
## 💡 Seção 10: Discussão

### Interpretação Física dos Resultados

1. **Tempo de residência hidráulica:** O parâmetro $\tau = V/Q = 100$ min é a escala temporal natural do sistema. Após ~3τ (300 min), o reator atinge 95% do regime permanente. Isso significa que o reator "esquece" a condição inicial após ~3 tempos de residência.

2. **Número de Damköhler:** $Da = k\tau = 10$ indica que a reação é 10× mais rápida que a renovação do fluido. Para $Da \gg 1$, a eficiência de remoção é alta (90,9%). Para $Da \ll 1$, o reator se aproxima de um simples tanque de passagem.

3. **Concentração de regime permanente:** $C_{ss} = C_{in}/(1+k\tau) = 4{,}55$ mg/L. A eficiência de remoção de 90,9% é adequada para muitos padrões de lançamento.

4. **Método de Euler:** Com Δt = 1 min, o erro é desprezível (< 0,01 mg/L). Para problemas com $k$ muito elevado ou $\tau$ muito pequeno, pode ser necessário reduzir Δt ou usar métodos de ordem superior.

### Limitações do Modelo

- **Mistura perfeita:** Assume concentração uniforme. Na realidade, há zonas mortas e curtos-circuitos.
- **Reação de 1ª ordem:** A cinética real pode ser mais complexa (Michaelis-Menten, inibição por substrato).
- **Volume constante:** Não considera evaporação ou variação de nível.
- **C_in constante:** Em aplicações reais, a concentração de entrada varia com o tempo.
- **Isotérmico:** Não considera efeito da temperatura na constante de reação.

### Aplicações Práticas

- **Tratamento de efluentes:** Dimensionamento de reatores biológicos (lodo ativado, biofilmes).
- **Indústria química:** Projeto de reatores de polimerização e síntese.
- **Farmacêutica:** Modelagem de reatores de fermentação.
- **Meio ambiente:** Modelagem de lagos e reservatórios como reatores CSTR.

### Conexão com a Teoria

> 📖 **Referência:** O Teorema do Transporte de Reynolds é apresentado em Lugon Jr. et al. (2026), Cap. 3, Seções 3.1–3.2. A modelagem de reatores CSTR segue Fogler (2020), Cap. 4. O método de Euler e análise de estabilidade são descritos em Lugon Jr. et al. (2026), Cap. 4, Seção 4.2. A aplicação a transporte de poluentes segue Schnoor (1996), Cap. 2.

---
## 🔄 Seção 11: Navegação

- [📚 Voltar ao Capítulo 3](../notebooks/03_Modelagem_Matematica.ipynb)
- [📂 Outros Estudos de Caso](README.md)
- [🏠 Repositório Principal](../README.md)

---

<div align="center">
Estudo de Caso 3.1

Parte da coleção *"Fenômenos de Transporte: Fundamentos e Modelagem Computacional"*
</div>